In [3]:
import pandas as pd
import os

In [5]:
# File path where your NHANES data is stored
file_path = "/Users/geetham/Documents/Files/"

In [11]:
pip install pyreadstat

Note: you may need to restart the kernel to use updated packages.


In [25]:
import pandas as pd

# Load the Body Measures data
bmx_data = pd.read_sas('/Users/geetham/Documents/Files/BMX_L.xpt')

# Load the Demographic data
demo_data = pd.read_sas('/Users/geetham/Documents/Files/DEMO_L.xpt')

# Load the Depression data
dpq_data = pd.read_sas('/Users/geetham/Documents/Files/DPQ_L.xpt')

# Merge the datasets on the respondent ID
merged_data = pd.merge(demo_data, dpq_data, on='SEQN', how='inner')
merged_data = pd.merge(merged_data, bmx_data, on='SEQN', how='inner')

# Display all column names to inspect the dataset
merged_data.columns

Index(['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN',
       'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMDBORN4',
       'DMDYRUSR', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'DMDHHSIZ', 'DMDHRGND',
       'DMDHRAGZ', 'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ', 'WTINT2YR', 'WTMEC2YR',
       'SDMVSTRA', 'SDMVPSU', 'INDFMPIR', 'DPQ010', 'DPQ020', 'DPQ030',
       'DPQ040', 'DPQ050', 'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090', 'DPQ100',
       'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD',
       'BMIHEAD', 'BMXHT', 'BMIHT', 'BMXBMI', 'BMDBMIC', 'BMXLEG', 'BMILEG',
       'BMXARML', 'BMIARML', 'BMXARMC', 'BMIARMC', 'BMXWAIST', 'BMIWAIST',
       'BMXHIP', 'BMIHIP'],
      dtype='object')

In [35]:
# Create PHQ-9 score and binary depression variable
phq9_columns = ['DPQ010', 'DPQ020', 'DPQ030', 'DPQ040', 'DPQ050', 'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090']
merged_data['PHQ9_Score'] = merged_data[phq9_columns].sum(axis=1)
merged_data['Depressed'] = (merged_data['PHQ9_Score'] >= 10).astype(int)

# Select and rename columns in one step
filtered_data = merged_data[[
    'RIDAGEYR', 'RIAGENDR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR', 'BMXBMI',
    'PHQ9_Score', 'Depressed', 'DMDHHSIZ'
]].rename(columns={
    'RIDAGEYR': 'Age',
    'RIAGENDR': 'Gender',
    'RIDRETH1': 'Race_Ethnicity',
    'DMDEDUC2': 'Education',
    'INDFMPIR': 'Poverty_Ratio',
    'BMXBMI': 'BMI',
    'DMDHHSIZ': 'Household_Size'
})

# Display missing values
missing_values = filtered_data.isnull().sum()
print(missing_values)

Age                 0
Gender              0
Race_Ethnicity      0
Education         273
Poverty_Ratio     831
BMI               102
PHQ9_Score          0
Depressed           0
Household_Size      0
dtype: int64


In [37]:
# Create depression score (sum of PHQ-9 items)
merged_data['phq9_score'] = merged_data[['DPQ010', 'DPQ020', 'DPQ030', 'DPQ040', 'DPQ050', 'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090']].sum(axis=1)

# Create binary depression variable (1 if depressed)
merged_data['depressed'] = (merged_data['phq9_score'] >= 10).astype(int)

# Create obesity variable (1 if BMI ≥ 30)
merged_data['obese'] = (merged_data['BMXBMI'] >= 30).astype(int)

Shape after dropping missing values: (5194, 9)

Missing values after cleaning:
Age               0
Gender            0
Race_Ethnicity    0
Education         0
Poverty_Ratio     0
BMI               0
PHQ9_Score        0
Depressed         0
Household_Size    0
dtype: int64


In [39]:
# Create binary outcome variable for Obesity based on BMI ≥ 30
filtered_data['Obese'] = (filtered_data['BMI'] >= 30).astype(int)

In [43]:
print(filtered_data['PHQ9_Score'].describe())
print(filtered_data['BMI'].describe())

count    6.337000e+03
mean     3.699858e+00
std      4.848740e+00
min      0.000000e+00
25%      4.857845e-78
50%      2.000000e+00
75%      5.000000e+00
max      5.100000e+01
Name: PHQ9_Score, dtype: float64
count    6235.000000
mean       29.659326
std         7.395826
min        11.100000
25%        24.500000
50%        28.300000
75%        33.400000
max        74.800000
Name: BMI, dtype: float64


In [45]:
# Remove invalid PHQ-9 scores
filtered_data = filtered_data[filtered_data['PHQ9_Score'] <= 27]

In [47]:
# Check value counts
print("Depression status counts:\n", filtered_data['Depressed'].value_counts())
print("\nObesity status counts:\n", filtered_data['Obese'].value_counts())

Depression status counts:
 0    5572
1     757
Name: Depressed, dtype: int64

Obesity status counts:
 0    3806
1    2523
Name: Obese, dtype: int64


In [57]:
from sklearn.linear_model import LogisticRegression

# Prepare data for propensity score estimation
# Drop rows with missing values in both covariates and depression status
filtered_data_clean = filtered_data.dropna(subset=['Depressed'] + covariates).copy()

# Separate covariates (X) and dependent variable (y)
X = filtered_data_clean[covariates]  # Covariates
y = filtered_data_clean['Depressed']  # Depression status

# Fit logistic regression model to estimate propensity scores
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X, y)

# Get propensity scores
propensity_scores = logreg.predict_proba(X)[:, 1]  # Probability of being in 'Depressed' group (1)

# Add propensity scores to the dataset using .loc to avoid SettingWithCopyWarning
filtered_data_clean.loc[:, 'Propensity_Score'] = propensity_scores

In [77]:
from sklearn.neighbors import NearestNeighbors

# Match treated (depressed = 1) and control (depressed = 0) individuals based on propensity scores
treated = filtered_data_clean[filtered_data_clean['Depressed'] == 1]
control = filtered_data_clean[filtered_data_clean['Depressed'] == 0]

# Fit nearest neighbors model to find closest matches for each treated individual
knn = NearestNeighbors(n_neighbors=1)
knn.fit(control[['Propensity_Score']])  # Match on propensity score

# For each treated individual, find the nearest control individual
distances, indices = knn.kneighbors(treated[['Propensity_Score']])

# Get matched control individuals
matched_controls = control.iloc[indices.flatten()]

# Combine the treated and matched control individuals
matched_data = pd.concat([treated, matched_controls])

# Round the data to 3 decimal places
matched_data_rounded = matched_data.round(3)

# Format and display the summary statistics with 3 decimal places
pd.set_option('display.float_format', '{:,.3f}'.format)

# Print the rounded summary statistics
print(matched_data_rounded.describe())

            Age    Gender  Race_Ethnicity  Education  Poverty_Ratio       BMI  \
count 1,212.000 1,212.000       1,212.000  1,212.000      1,212.000 1,212.000   
mean     49.648     1.625           3.090      3.630          2.149    30.955   
std      17.470     0.484           0.986      1.119          1.518     8.654   
min      20.000     1.000           1.000      1.000          0.000    15.200   
25%      33.750     1.000           3.000      3.000          0.960    24.700   
50%      51.000     2.000           3.000      4.000          1.750    29.400   
75%      65.000     2.000           3.000      4.000          3.072    35.425   
max      80.000     2.000           5.000      9.000          5.000    69.100   

       PHQ9_Score  Depressed  Household_Size     Obese  Propensity_Score  
count   1,212.000  1,212.000       1,212.000 1,212.000         1,212.000  
mean        8.256      0.500           2.395     0.461             0.159  
std         6.763      0.500           1.430 

In [83]:
# Calculate SMDs before matching (using absolute differences divided by pooled standard deviation)
def calc_smd(treated, control, covariate):
    mean_treated = treated[covariate].mean()
    mean_control = control[covariate].mean()
    pooled_sd = ((treated[covariate].std()**2 + control[covariate].std()**2) / 2)**0.5
    smd = abs(mean_treated - mean_control) / pooled_sd
    return round(smd, 2)  # Round the SMD to 2 decimal places

# List of covariates to check balance
covariates = ['Age', 'Gender', 'Race_Ethnicity', 'Education', 'Poverty_Ratio', 'BMI', 'Household_Size']

# Calculate SMDs for each covariate before matching
smd_before = {cov: calc_smd(treated, control, cov) for cov in covariates}
print("Standardized Mean Differences Before Matching:")
print(smd_before)

# Calculate SMDs after matching (using the matched data)
treated_matched = matched_data_rounded[matched_data_rounded['Depressed'] == 1]
control_matched = matched_data_rounded[matched_data_rounded['Depressed'] == 0]

smd_after = {cov: calc_smd(treated_matched, control_matched, cov) for cov in covariates}
print("Standardized Mean Differences After Matching:")
print(smd_after)

Standardized Mean Differences Before Matching:
{'Age': 0.3, 'Gender': 0.17, 'Race_Ethnicity': 0.02, 'Education': 0.26, 'Poverty_Ratio': 0.52, 'BMI': 0.19, 'Household_Size': 0.12}
Standardized Mean Differences After Matching:
{'Age': 0.04, 'Gender': 0.01, 'Race_Ethnicity': 0.05, 'Education': 0.0, 'Poverty_Ratio': 0.08, 'BMI': 0.06, 'Household_Size': 0.02}


In [85]:
# Calculate the mean outcome for treated and matched control individuals
mean_treated_outcome = treated_matched['BMI'].mean()  # Replace 'BMI' with your outcome variable
mean_control_outcome = control_matched['BMI'].mean()  # Replace 'BMI' with your outcome variable

# Calculate the ATT (difference in means)
att = mean_treated_outcome - mean_control_outcome
print(f"Average Treatment Effect on the Treated (ATT): {att:.2f}")

Average Treatment Effect on the Treated (ATT): 0.49


In [87]:
import numpy as np

# Bootstrapping to get confidence intervals
n_iterations = 1000
att_bootstrap = []

for _ in range(n_iterations):
    sample_treated = treated_matched.sample(frac=1, replace=True)
    sample_control = control_matched.sample(frac=1, replace=True)
    att_bootstrap.append(sample_treated['BMI'].mean() - sample_control['BMI'].mean())

# Calculate the 95% confidence interval
lower_ci = np.percentile(att_bootstrap, 2.5)
upper_ci = np.percentile(att_bootstrap, 97.5)
print(f"95% Confidence Interval for ATT: ({lower_ci:.2f}, {upper_ci:.2f})")

95% Confidence Interval for ATT: (-0.45, 1.50)


In [89]:
from scipy.stats import ttest_rel

# Ensure both groups are aligned index-wise (since they're 1:1 matched)
treated_outcome = treated_matched['Obese'].reset_index(drop=True)
control_outcome = control_matched['Obese'].reset_index(drop=True)

# Perform paired t-test
t_stat, p_value = ttest_rel(treated_outcome, control_outcome)

print(f"T-statistic: {round(t_stat, 2)}")
print(f"P-value: {round(p_value, 4)}")

# Interpretation
if p_value < 0.05:
    print("Result is statistically significant (p < 0.05)")
else:
    print("Result is not statistically significant (p ≥ 0.05)")

T-statistic: 1.12
P-value: 0.2641
Result is not statistically significant (p ≥ 0.05)


In [91]:
import statsmodels.api as sm
X = sm.add_constant(matched_data_rounded['Depressed'])
model = sm.OLS(matched_data_rounded['Obese'], X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Obese   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.198
Date:                Tue, 15 Apr 2025   Prob (F-statistic):              0.274
Time:                        15:05:37   Log-Likelihood:                -875.40
No. Observations:                1212   AIC:                             1755.
Df Residuals:                    1210   BIC:                             1765.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.4455      0.020     21.995      0.0

In [95]:
import statsmodels.api as sm
import numpy as np

# Use your existing DataFrame (update the name if different)
data = matched_data_rounded  # Replace with your actual DataFrame if needed

# Define treatment and outcome
T = data['Depressed']
Y = data['Obese']
ps = data['Propensity_Score']

# Stabilized IPTW weights
treated_weight = T / ps
control_weight = (1 - T) / (1 - ps)
iptw_weights = treated_weight + control_weight

# Add intercept for OLS
X = sm.add_constant(T)

# Weighted regression model (ATT estimate)
iptw_model = sm.WLS(Y, X, weights=iptw_weights)
iptw_result = iptw_model.fit()

# Display results
print("\nIPTW Weighted Regression Results:")
print(iptw_result.summary())


IPTW Weighted Regression Results:
                            WLS Regression Results                            
Dep. Variable:                  Obese   R-squared:                       0.000
Model:                            WLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.3715
Date:                Tue, 15 Apr 2025   Prob (F-statistic):              0.542
Time:                        15:06:56   Log-Likelihood:                -1174.1
No. Observations:                1212   AIC:                             2352.
Df Residuals:                    1210   BIC:                             2362.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.

In [99]:
# Assign your working dataset to df for consistency
df = matched_data_rounded

# Now run logistic regression with covariate adjustment
import statsmodels.api as sm

X = df[['Depressed', 'Age', 'Gender', 'Race_Ethnicity', 'Education', 'Poverty_Ratio', 'Household_Size']]
X = sm.add_constant(X)
y = df['Obese']

logit_model = sm.Logit(y, X).fit()
print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.675584
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:                  Obese   No. Observations:                 1212
Model:                          Logit   Df Residuals:                     1204
Method:                           MLE   Df Model:                            7
Date:                Tue, 15 Apr 2025   Pseudo R-squ.:                 0.02109
Time:                        15:08:09   Log-Likelihood:                -818.81
converged:                       True   LL-Null:                       -836.45
Covariance Type:            nonrobust   LLR p-value:                 9.924e-06
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -1.7243      0.415     -4.154      0.000      -2.538      -0.911
Depressed        